In [3]:
!conda run -n torsional_diffusion python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

2.8.0+cu128 True



In [4]:
%cd ~/AttUnode/dataset/torsional-diffusion

/root/AttUnode/dataset/torsional-diffusion


In [5]:
with open("caffeine.csv", "w") as f:
    f.write("smiles,n_confs,smiles\n")
    f.write("CN1C=NC2=C1C(=O)N(C(=O)N2C)C,10,CN1C=NC2=C1C(=O)N(C(=O)N2C)C\n")

In [6]:
!conda run -n torsional_diffusion python generate_confs.py \
    --test_csv caffeine.csv \
    --inference_steps 20 \
    --model_dir workdir/workdir/drugs_default \
    --confs_per_mol 10 \
    --out caffeine_torsional_diffusion_confs.pkl \
    --tqdm --batch_size 128 --no_energy

CN1C=NC2=C1C(=O)N(C(=O)N2C)C
0 rotable_bonds=3 n_confs=10 CN1C=NC2=C1C(=O)N(C(=O)N2C)C
Generated conformers for 1 molecules


100%|██████████| 1/1 [00:04<00:00,  4.63s/it]



In [7]:
!pip install rdkit py3Dmol -q

import py3Dmol
from rdkit import Chem

supplier = Chem.SDMolSupplier("caffeine_torsional_diffusion_conformers.sdf")
mols = [m for m in supplier if m is not None]

view = py3Dmol.view(viewergrid=(2, 5), width=1000, height=400, linked=False)
for i, mol in enumerate(mols):
    molblock = Chem.MolToMolBlock(mol)
    row, col = divmod(i, 5)
    view.addModel(molblock, "mol", viewer=(row, col))
    view.setStyle({'stick': {}, 'sphere': {'scale': 0.25}}, viewer=(row, col))
    view.zoomTo(viewer=(row, col))
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd

# Load the 10 conformers saved earlier
supplier = Chem.SDMolSupplier("caffeine_torsional_diffusion_conformers.sdf", removeHs=False)
mols = [m for m in supplier if m is not None]

names = []
energies = []

for mol in mols:
    name = mol.GetProp("_Name") if mol.HasProp("_Name") else "unnamed"
    props = AllChem.MMFFGetMoleculeProperties(mol)
    ff = AllChem.MMFFGetMoleculeForceField(mol, props)
    energy = ff.CalcEnergy()   # single-point energy, no optimization — reflects the model's raw output geometry
    names.append(name)
    energies.append(energy)

df = pd.DataFrame({"Conformer": names, "Energy (kcal/mol)": energies})
df = df.sort_values("Energy (kcal/mol)").reset_index(drop=True)
print(df.to_string(index=False))

                          Conformer  Energy (kcal/mol)
caffeine_torsional_diffusion_conf_4        -108.078839
caffeine_torsional_diffusion_conf_0        -107.361384
caffeine_torsional_diffusion_conf_5        -106.637447
caffeine_torsional_diffusion_conf_7        -104.985390
caffeine_torsional_diffusion_conf_9        -104.227626
caffeine_torsional_diffusion_conf_3        -104.017726
caffeine_torsional_diffusion_conf_1        -102.942092
caffeine_torsional_diffusion_conf_6        -102.939170
caffeine_torsional_diffusion_conf_8        -102.849772
caffeine_torsional_diffusion_conf_2        -102.734025
